
# FilterBundle: Batched Convolution for MCMC

MCMC inference evaluates a forward model millions of times.  If each evaluation must
convolve a spectral energy distribution through several photometric filters, the per-filter
loop cost accumulates quickly.

:class:`~trilobite.utils.phot_utils.FilterBundle` solves this by precomputing a
**weight matrix** $W$ of shape ``(N_\mathrm{filters},\, N_\mathrm{common})`` at
construction time.  During inference the band-averaged fluxes for *all* filters reduce to a
single matrix–vector multiply:

\begin{align}\mathbf{F}_\mathrm{bands} = W \cdot \mathbf{F}_\nu\end{align}

This example demonstrates:

1. Assembling a :class:`~trilobite.utils.phot_utils.FilterBundle` from individual filters.
2. Inspecting the **common frequency grid** and **weight matrix**.
3. The MCMC-optimised ``apply()`` workflow vs. the convenience ``convolve_nu()`` path.
4. Batched evaluation over a time series of SEDs in one call.
5. A timing comparison that quantifies the speedup over a naive filter loop.

## Relevant API references
- :class:`trilobite.utils.phot_utils.FilterBundle`
- :meth:`trilobite.utils.phot_utils.FilterBundle.apply`
- :meth:`trilobite.utils.phot_utils.FilterBundle.convolve_nu`


## Imports



In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import astropy.units as u

from trilobite.utils.phot_utils import FilterBundle, PhotometryFilter, flux_to_ab_mag
from trilobite.utils.plot_utils import set_plot_style

set_plot_style()

## Build a Set of Filters

We approximate the ZTF *g*, *r*, and *i* passbands as Gaussians
and add a broad *z*-band to illustrate overlapping filter ranges.



In [ ]:
BANDS = {
    "g": (4720, 650),
    "r": (6340, 820),
    "i": (7890, 900),
    "z": (9000, 1100),
}

filters = {}
for band, (center, width) in BANDS.items():
    lam = np.linspace(center - 2.5 * width, center + 2.5 * width, 300) * u.AA
    T = np.exp(-0.5 * ((lam.value - center) / (width / 2.35)) ** 2)
    filters[band] = PhotometryFilter(lam, T, name=f"ztf-{band}")

## Constructing the FilterBundle

Passing the dict to :class:`~trilobite.utils.phot_utils.FilterBundle` triggers
construction of the common frequency grid and weight matrix.



In [ ]:
bundle = FilterBundle(filters)
print(bundle)

## Inspecting the Common Grid and Weight Matrix

The common grid is the sorted union of all individual filter grids.  The weight
matrix has one row per filter; each row sums to 1 so that ``W @ F_nu`` gives
band-averaged flux densities directly.



In [ ]:
print(f"\nCommon grid size  : {len(bundle.frequency_grid):,} points")
print(f"Frequency range   : {bundle.frequency_grid.min():.3e}–{bundle.frequency_grid.max():.3e} Hz")
print(f"Weight matrix shape: {bundle.weight_matrix.shape}")
print(f"Row sums (should all be 1): {bundle.weight_matrix.sum(axis=1)}")

## Visualise the Weight Matrix

Each row of the weight matrix corresponds to one filter's contribution at
each point on the common grid.



In [ ]:
lam_common_aa = (2.99792458e10 / bundle.frequency_grid) * 1e8  # cm → Å  (descending)

fig, axes = plt.subplots(len(filters), 1, figsize=(10, 7), sharex=True)
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(filters)))

for ax, color, name, row in zip(axes, colors, bundle.filter_names, bundle.weight_matrix):
    ax.fill_between(lam_common_aa, row, color=color, alpha=0.6)
    ax.set_ylabel(name, fontsize=9)
    ax.set_ylim(bottom=0)

axes[-1].set_xlabel("Wavelength [Å]")
fig.suptitle("FilterBundle weight matrix rows on the common grid")
plt.tight_layout()
plt.show()

## The MCMC Hot-Loop Pattern

The recommended MCMC workflow has two phases:

**Setup (once before sampling):**
Obtain the common evaluation grid from the bundle and pre-allocate anything
that depends only on the filter configuration.

**Inference (called millions of times):**
Evaluate the SED model on the common grid, then call
:meth:`~trilobite.utils.phot_utils.FilterBundle.apply` — a pure matrix
multiply with no interpolation overhead.



In [ ]:
# Setup phase ---------------------------------------------------------------
nu_eval = bundle.frequency_grid  # shape (N_common,) — the grid to evaluate on
c_cgs = 2.99792458e10


def power_law_sed(nu, F_ref, alpha, nu_ref=1e14):
    """Simple power-law SED: F_nu = F_ref * (nu/nu_ref)^alpha."""
    return F_ref * (nu / nu_ref) ** alpha


# Inference phase (single SED) ---------------------------------------------
theta = {"F_ref": 3.631e-20, "alpha": -0.8}

F_nu_model = power_law_sed(nu_eval, **theta)  # shape (N_common,)
F_bands = bundle.apply(F_nu_model)  # shape (N_filters,)  ← one matrix multiply

print("\nBand-averaged fluxes for a single SED:")
for name, F in zip(bundle.filter_names, F_bands):
    print(f"  {name}:  {F:.4e} erg/s/cm²/Hz  →  AB mag {flux_to_ab_mag(F):.2f}")

## Batched Evaluation: Multi-Epoch Time Series

Passing a 2-D array ``F_nu`` of shape ``(N_t, N_common)`` returns
``(N_t, N_filters)`` in a single call — no Python loop needed.



In [ ]:
N_t = 50  # number of time steps
t_days = np.linspace(1, 100, N_t)

# Toy model: rising then fading power law with evolving spectral index
F_ref_t = 1e-23 * (t_days / 10) ** 0.5 * np.exp(-t_days / 80)
alpha_t = -0.5 - 0.3 * (t_days / 100)

# Build (N_t, N_common) flux array
F_nu_batch = np.outer(F_ref_t, (nu_eval / 1e14) ** alpha_t.mean())
# (for a proper per-epoch spectral index we broadcast:)
F_nu_batch = F_ref_t[:, None] * (nu_eval[None, :] / 1e14) ** alpha_t[:, None]

F_bands_batch = bundle.apply(F_nu_batch)  # (N_t, N_filters)
mags_batch = flux_to_ab_mag(F_bands_batch)  # (N_t, N_filters)

print(f"\nBatched result shape: {F_bands_batch.shape}  (N_t={N_t}, N_filters={len(bundle)})")

## Plot the Synthetic Multi-Band Light Curve



In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.plasma(np.linspace(0.1, 0.85, len(bundle)))

for color, name, mags in zip(colors, bundle.filter_names, mags_batch.T):
    ax.plot(t_days, mags, color=color, lw=2, label=name)

ax.invert_yaxis()
ax.set_xlabel("Time since explosion [days]")
ax.set_ylabel("AB magnitude")
ax.set_title("Synthetic multi-band light curve from a FilterBundle")
ax.legend()
plt.tight_layout()
plt.show()

## Timing: Matrix Multiply vs. Filter Loop

We compare three approaches for a representative MCMC workload
(1 000 repeated single-SED evaluations):

1. :meth:`~trilobite.utils.phot_utils.FilterBundle.apply` — pure matrix multiply on the common grid.
2. :meth:`~trilobite.utils.phot_utils.FilterBundle.convolve_nu` — interpolation + matrix multiply.
3. Explicit Python loop over individual :class:`~trilobite.utils.phot_utils.PhotometryFilter` objects.



In [ ]:
N_reps = 1_000
F_test = power_law_sed(nu_eval, F_ref=3.631e-20, alpha=-0.7)

# Method 1: bundle.apply (pre-interpolated grid)
t0 = time.perf_counter()
for _ in range(N_reps):
    _ = bundle.apply(F_test)
t_apply = time.perf_counter() - t0

# Method 2: bundle.convolve_nu (arbitrary grid → interpolate then apply)
nu_arb = np.geomspace(2e13, 2e15, 600)
F_arb = power_law_sed(nu_arb, F_ref=3.631e-20, alpha=-0.7)

t0 = time.perf_counter()
for _ in range(N_reps):
    _ = bundle.convolve_nu(nu_arb, F_arb)
t_convolve = time.perf_counter() - t0

# Method 3: naive loop over individual filters
t0 = time.perf_counter()
for _ in range(N_reps):
    _ = [filt.convolve_nu(nu_arb, F_arb) for filt in bundle.filters.values()]
t_loop = time.perf_counter() - t0

print(f"\nTiming over {N_reps:,} evaluations ({len(bundle)} filters):")
print(f"  bundle.apply()       : {t_apply * 1e3:.2f} ms  ({t_loop / t_apply:.1f}x faster than loop)")
print(f"  bundle.convolve_nu() : {t_convolve * 1e3:.2f} ms  ({t_loop / t_convolve:.1f}x faster than loop)")
print(f"  filter loop          : {t_loop * 1e3:.2f} ms  (baseline)")

Visualise the speedup as a bar chart



In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
labels = ["bundle.apply()\n(pre-interpolated)", "bundle.convolve_nu()\n(arbitrary grid)", "filter loop\n(baseline)"]
times_ms = [t_apply * 1e3, t_convolve * 1e3, t_loop * 1e3]
colors_bar = ["#2ca02c", "#1f77b4", "#d62728"]

bars = ax.bar(labels, times_ms, color=colors_bar, width=0.5, edgecolor="black", linewidth=0.8)
ax.set_ylabel(f"Wall time for {N_reps:,} evaluations [ms]")
ax.set_title("FilterBundle vs. filter loop — timing comparison")

for bar, t in zip(bars, times_ms):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, f"{t:.1f} ms", ha="center", va="bottom", fontsize=10
    )

plt.tight_layout()
plt.show()

.. tip::

   For maximum MCMC throughput, evaluate your physical model directly on
   ``bundle.frequency_grid`` and call ``bundle.apply()`` inside the likelihood.
   This avoids the interpolation cost of ``convolve_nu()`` and typically delivers
   an order-of-magnitude speedup over per-filter loops.

